<a href="https://colab.research.google.com/github/allie-tran/202609-csc1159-resources/blob/main/notebooks/cleaning_data_with_pandas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Let's take another look at the data sampler set from Topic 1. I've made a sampler with a few sneaky errors and artefacts in it. We'll take a look at finding and fixing a few of them but which other ones can you find? 

In [1]:
import pandas as pd

First we need to read the file in (`read_csv`), check that it looks okay (`head`) and check what the data types looks like (`info`)

In [2]:
df = pd.read_csv("https://raw.githubusercontent.com/allie-tran/202609-csc1159-resources/main/data/sampler/DataSamplerMessy.csv")

Note that the url here is to a github repository and it's the direct link to the raw file. This lets us easily read the file in to the colab notebook without needing to upload the file. If you try this on your local computer you can either use the URL or download the .csv file and use a local filepath. 

In [3]:
df.head()

,Time,Float,Integer,Bigger,Fruit,ID,Ireland,Latitude,Longtitude
0,03/10/2016 16:00,0.662052,16.0,58.0,banana,AYPI,Rossan Pt.,54.700000,-8.783333
1,03/10/2016 16:01,0.123787,43.0,57.0,apple,RGVS,Mullet Pen,54.216667,-10.033333
2,03/10/2016 16:02,0.032207,62.0,66.0,banana,JOBI,Erne,54.500000,-8.266667
3,03/10/2016 16:03,0.865759,78.0,60.0,celery,MBPL,Kilrush,52.633333,-9.483333
4,03/10/2016 16:04,0.321649,78.0,52.0,banana,MFRO,Louth,53.083333,-6.533333


Notice how the `read_csv` command has assumed that the first row is the column headers. No need to specify it this time!

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 152 entries, 0 to 151
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Time        152 non-null    object 
 1   Float       151 non-null    float64
 2   Integer     151 non-null    float64
 3   Bigger      151 non-null    float64
 4   Fruit       151 non-null    object 
 5   ID          151 non-null    object 
 6   Ireland     151 non-null    object 
 7   Latitude    151 non-null    float64
 8   Longtitude  151 non-null    float64
dtypes: float64(5), object(4)
memory usage: 10.8+ KB


Everything looks okay but some of the Dtypes don't match what we'd like to have and the index of the dataframe is just the row number. It would be much nicer to have the data indexed by the timestamp. Let's change the type of the "Time" column to datetime (`to_datetime`) and set that column to be the frame index.

In [5]:
pd.to_datetime(df['Time']).head()   # Let's check that the conversion works first ...


0   2016-03-10 16:00:00
1   2016-03-10 16:01:00
2   2016-03-10 16:02:00
3   2016-03-10 16:03:00
4   2016-03-10 16:04:00
Name: Time, dtype: datetime64[ns]

Oh no! It's used American date formats! The date should be the 3rd of October not the 10th of March. This is a common data error. Let's fix this using the `dayfirst=True` argument. 

In [6]:
pd.to_datetime(df['Time'], dayfirst=True).head()

0   2016-10-03 16:00:00
1   2016-10-03 16:01:00
2   2016-10-03 16:02:00
3   2016-10-03 16:03:00
4   2016-10-03 16:04:00
Name: Time, dtype: datetime64[ns]

That's better. Now let's change the column in the original dataframe by assigning the output from to_datetime to column 'Time' and check the data types again.

In [7]:
df['Time'] = pd.to_datetime(df['Time'], dayfirst=True)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 152 entries, 0 to 151
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   Time        152 non-null    datetime64[ns]
 1   Float       151 non-null    float64       
 2   Integer     151 non-null    float64       
 3   Bigger      151 non-null    float64       
 4   Fruit       151 non-null    object        
 5   ID          151 non-null    object        
 6   Ireland     151 non-null    object        
 7   Latitude    151 non-null    float64       
 8   Longtitude  151 non-null    float64       
dtypes: datetime64[ns](1), float64(5), object(3)
memory usage: 10.8+ KB


See that column 'Time' is now datetime64 format that will let us apply [datetime specific functions](https://pandas.pydata.org/pandas-docs/stable/user_guide/timeseries.html) to this column. I'm going to set the Time column to be the index of the dataframe.

In [8]:
df = df.set_index('Time')
df.head()

,Float,Integer,Bigger,Fruit,ID,Ireland,Latitude,Longtitude
Time,,,,,,,,
2016-10-03 16:00:00,0.662052,16.0,58.0,banana,AYPI,Rossan Pt.,54.700000,-8.783333
2016-10-03 16:01:00,0.123787,43.0,57.0,apple,RGVS,Mullet Pen,54.216667,-10.033333
2016-10-03 16:02:00,0.032207,62.0,66.0,banana,JOBI,Erne,54.500000,-8.266667
2016-10-03 16:03:00,0.865759,78.0,60.0,celery,MBPL,Kilrush,52.633333,-9.483333
2016-10-03 16:04:00,0.321649,78.0,52.0,banana,MFRO,Louth,53.083333,-6.533333


A useful option now that we have a datetime index is we can get the values between 4:55pm and 5:05pm using the between_times() function. 

In [9]:
df.between_time('16:55', '17:05')

,Float,Integer,Bigger,Fruit,ID,Ireland,Latitude,Longtitude
Time,,,,,,,,
2016-10-03 16:55:00,0.824485,30.0,60.0,banana,NOTM,Inishmaan,53.083333,-9.583333
2016-10-03 16:56:00,0.471286,64.0,58.0,banana,BTND,Finn,54.850000,-7.466667
2016-10-03 16:57:00,0.903258,18.0,58.0,banana,KVHA,Boyne,53.716667,-6.250000
2016-10-03 16:58:00,0.832011,49.0,57.0,banana,SFEP,Clones,54.183333,-7.250000
2016-10-03 16:59:00,0.173285,58.0,59.0,apple,QDMN,Castleblaney,54.116667,-6.733333
2016-10-03 17:00:00,0.910233,54.0,63.0,banana,QBNW,Cootehill,54.066667,-7.083333
2016-10-03 17:01:00,0.459457,32.0,58.0,berry,WMNQ,Rath Luirc,52.350000,-8.666667
2016-10-03 17:02:00,0.121816,87.0,66.0,orange,XEQE,Castlerea,53.766667,-8.483333
2016-10-03 17:03:00,0.450045,82.0,66.0,apple,SSBZ,Malin Pen,55.333333,-7.283333


# Starting to clean: Duplicates  

Now that we have the data set loaded let's try checking and cleaning.

Take a look at the documentation for [.duplicated](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.duplicated.html?highlight=duplicated#pandas.DataFrame.duplicated). Note that it returns a Series -- a list of Index : Boolean values where True shows the rows that are complete duplicates. You'll also need to check the Index for duplicates and to use the subset argument.

In [10]:
df.duplicated()

Time
2016-10-03 16:00:00    False
2016-10-03 16:01:00    False
2016-10-03 16:02:00    False
2016-10-03 16:03:00    False
2016-10-03 16:04:00    False
                       ...  
2016-10-03 18:27:00    False
2016-10-03 18:28:00    False
2016-10-03 18:28:00     True
2016-10-03 18:29:00    False
2016-10-03 18:30:00    False
Length: 152, dtype: bool

To see the rows that are duplicated use the result from df.duplicated() as a mask for the dataframe. 

In [11]:
df[df.duplicated()]

,Float,Integer,Bigger,Fruit,ID,Ireland,Latitude,Longtitude
Time,,,,,,,,
2016-10-03 18:28:00,0.571263,51.0,66.0,apple,SVQR,Kenmare,51.883333,-9.6


So the entry at 6:28pm is a completely duplicate row. Let's drop that.

In [12]:
df_clean = df.drop_duplicates()

If you remember when we checked the info for the dataframe there were 152 entries. Now that we've dropped the one duplicate there should be 151. Let's check. We're also now working on a copy of the data (`df_clean`) since we are going to be making changes.

In [13]:
df_clean.info()  # or you could use .shape

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 151 entries, 2016-10-03 16:00:00 to 2016-10-03 18:30:00
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Float       150 non-null    float64
 1   Integer     150 non-null    float64
 2   Bigger      150 non-null    float64
 3   Fruit       150 non-null    object 
 4   ID          150 non-null    object 
 5   Ireland     150 non-null    object 
 6   Latitude    150 non-null    float64
 7   Longtitude  150 non-null    float64
dtypes: float64(5), object(3)
memory usage: 10.6+ KB


Using duplicated on the whole dataframe only checks for completely duplicated rows. Let's also check that the index is unique and there's a column titled 'ID' which sounds like it should be unique too. 

In [14]:
df_clean[df_clean.index.duplicated()]

,Float,Integer,Bigger,Fruit,ID,Ireland,Latitude,Longtitude
Time,,,,,,,,


In [15]:
df_clean[df_clean.duplicated(subset=['ID'], keep=False)]   # note the subset and the keep arguments

,Float,Integer,Bigger,Fruit,ID,Ireland,Latitude,Longtitude
Time,,,,,,,,
2016-10-03 16:31:00,0.086533,77.0,64.0,banana,SSBZ,Slieve Authty,53.066667,-8.500000
2016-10-03 17:03:00,0.450045,82.0,66.0,apple,SSBZ,Malin Pen,55.333333,-7.283333


See how we can use the subet and keep arguments to get the rows that have a duplicate ID. The ID column is nonsense in this dataset but what might you do if you had to repair a duplicate ID? I'm going to change the second entry to SSCZ since I know that's not already in use here.

In [16]:
df_clean.loc["2016-10-03 17:03:00", "ID"] = "SSCZ"

To change one instance in Pandas is oddly tricky. Using `.loc[row_index, column_name] = "new value"` will mostly work though it's not ideal and you may get a warning. In this case this is because we are working on a copy of df (`df_clean`) so the assignment of the new ID value will still work. Let's check.

In [17]:
df_clean.loc["2016-10-03 17:03:00"]

Float          0.450045
Integer            82.0
Bigger             66.0
Fruit             apple
ID                 SSCZ
Ireland       Malin Pen
Latitude      55.333333
Longtitude    -7.283333
Name: 2016-10-03 17:03:00, dtype: object

In [18]:
df_clean[df_clean.duplicated(subset=['ID'], keep=False)]

,Float,Integer,Bigger,Fruit,ID,Ireland,Latitude,Longtitude
Time,,,,,,,,


So the duplicate ID has been fixed and there are no other duplicate IDs in the sampler dataset. 

# Missing values?

In Pandas [missing values](https://pandas.pydata.org/pandas-docs/stable/user_guide/missing_data.html) are also referred to as NA. There are many options for handling these. First let's see if we have any in this dataset.


In [19]:
df_clean[df_clean.ID.isna()]   # here I'm just checking the ID column for NaN

,Float,Integer,Bigger,Fruit,ID,Ireland,Latitude,Longtitude
Time,,,,,,,,
2016-10-03 16:14:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Looks like we have lost data that has been read and had the values recorded as NaN (missing value). For the moment we don't know how to fix this so let's go on and check the place names to see if there are are missing. It's dangerous to replace a fully missing row. In reality you'd probably need to go back to the data source and try to find the information. That is why Pandas will flag missing values with the special NaN/NaT entries. 

# Are the place names correct?

How can we check the content of the Ireland column for errors in place names? This can be a difficult one but luckily here we have a reference list! Use the dataset at https://raw.githubusercontent.com/allie-tran/202609-csc1159-resources/main/data/sampler/ireland_locations.xlsx. Not that this is an Excel file (`read_excel`). 

You can then use the [.isin function](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.isin.html) to check if all the required locations in Ireland are correctly included in the dataset. This will return `True` for all values that are correct so use `~` to get the inverse values and show the entries that are not in the reference list. How many do you find?

# Whitespace

There's a lot of weird leading and trailing whitespace in the "Ireland" column! Use functions like  [`str.strip()`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.Series.str.strip.html), `str.lstrip()` and `str.rstrip()`. Note that here you want to include the str. as part of the function name when applying the function to the Series. 

# What about the Fruit column?

Take a look at the Fruit column and see if you can spot anything odd. Start by getting the unique values using the `unique()` function.

What errors or odd entries do you find? Can you use the `.loc[row_index, column_name] = value` method to fix some of them?

You might be interested to know that some entries were included for onion that should have been pear. How odd. You can replace all instances of a value using the `.replace('onion', 'pear')` function. Check the whitespace first!

# Final thoughts

This notebook gives just a few examples of the types of functions you can use to explore and clean data using the Pandas library. There are similar functions in R. Compare these to the activities you did using Google Sheets and OpenRefine, which do you think are easier, more useful or more powerful? 